# Case set up
### Import libraries

In [ ]:
import os
import sys
import copy
import time
import importlib
import numpy as np
import joblib as jl
import tensorflow as tf

from tqdm import tqdm

Visual set up

In [ ]:
import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
%matplotlib notebook

# Use custom style
style_name = 'paper'
mpl.style.use(
    os.environ['WORKSPACE_PATH'] + '/RESEARCH/styles/matplotlib/' + style_name + '.mplstyle'
) # See: https://matplotlib.org/1.5.3/users/style_sheets.html

In [ ]:
# Get colors list
colors = mpl.rcParams['axes.prop_cycle'].by_key()['color']

# Figure/animation saving parameters
# mpl.rcParams['savefig.dpi'] = 100

# Font parameters
# mpl.rcParams['text.latex.preamble'].join([
#     r'\usepackage{amsmath}',
#     r'\boldmath',
#     r'\usepackage{pifont}',
#     r'\usepackage{tikz}',
#     r'\usepackage[utf8]{inputenc}'
# ])
# mpl.rcParams['font.family'] = 'lmodern'

# # Tick parameters
# mpl.rcParams['xtick.labelsize'] = 40
# mpl.rcParams['ytick.labelsize'] = 40
# mpl.rcParams['xtick.major.size'] = 20
# mpl.rcParams['xtick.major.width'] = 2
# mpl.rcParams['xtick.minor.size'] = 10
# mpl.rcParams['xtick.minor.width'] = 2

# # Tick parameters
# mpl.rcParams['axes.labelsize'] = 40
# mpl.rcParams['axes.titlesize'] = 40
# mpl.rcParams['legend.fontsize'] = 35


### Define PrODE and working case directories paths

In [ ]:
prode_dir = os.path.normpath(os.environ['WORKSPACE_PATH'] + '/ML/PrODE/prode/')
working_dir = "./"

In [ ]:
if importlib.util.find_spec('prode') is None:
    sys.path.append(prode_dir)
sys.path.append(working_dir)

### Import additional libraries

In [ ]:
# PrODE -----------------------------------------------------------------------
from prode.utilities import utils
from prode.apps import functionObjects as fun
from prode.postprocessing import testing, training

# Extra modules ---------------------------------------------------------------
# Inputs
import inputs
# Physics-related
from physics.data_generator import DataGenerator
from prode.physics.general import const
from prode.physics.thermochemistry import specie as specieMdl
from prode.physics.thermochemistry.reactionThermo import mixture as mixMdl

### Initialize network

In [ ]:
net = fun._net(
    input_module=inputs
)

### Initialize PrODE model

In [ ]:
from inputs.general import flags

for k in ('train', 'save'):
    flags[k] = 0
if flags['load'] not in (1, 2):
    utils.raise_value_err(
        "The network has to be restored from a saved or a checkpointed "
        "one. Modified the 'load' entry in the `flags` input variable."
    )
model = fun._model(
    flags=flags,
    input_module=inputs,
    net=net
)

### Generate testing data

In [ ]:
# Generate Testing Data ---------------------------------------------------
dataGen_args = utils.get_args_values(
    args=utils.get_args(DataGenerator.__init__),
    modules=[inputs.data, inputs.general]
)
dataGen = DataGenerator(
    **dataGen_args
)
dataGen.data_dir = "./data/"
if not os.path.exists(dataGen.data_dir):
    os.makedirs(dataGen.data_dir)

utils.print_main("Generating testing pts ...")
if inputs.data.from_file:
    test_data = dataGen.testing_data(
        inputs.data.file['test'],
        header=inputs.data.header
    )
else:
    raise NotImplementedError

# Postprocessing - Model analysis

## Set up analysis

In [ ]:
path = './modelAnalysis/'
if not os.path.exists(path):
    os.makedirs(path)
    
FLAGS = {
    'pred': 0,
    'one_pass': 0,
    'speedup': 0,
    'recursive': 0,
    'noneq': 0,
    'levels': 0,
    'animation': 0
}

#### Get predictions

In [ ]:
if FLAGS['pred']:
    utils.print_main("Postprocessing physical quantities ...")
    x, y, y_pred = [], [], []
    for scenario in tqdm(test_data):
        x_i, y_i = scenario
        x.append(x_i)
        y.append(y_i)
        y_pred.append(model.net.predict(x_i))
    n = len(test_data)

## One-pass inference
Setting up

In [ ]:
path_ = path + '/onePassInference/'
if not os.path.exists(path_):
    os.makedirs(path_)

ts_label = {}
for i, name in enumerate(inputs.network.names):
    d = 1 if i == 0 else inputs.general.dim['out'][i]
    ts_label[name] = [r'$O$'] \
        + [r'$O_2^{({%s},{%s})}$' % (i+1,j+1) for j in range(d)]

In [ ]:
def output_ops(y, timescale):
    if timescale == 0:
        return tf.reverse(y[0], axis=[1])
    else:
        return tf.concat([y[0][:,1:2], y[timescale]], 1)

def single_scenario(path, i):
    path_case = path_+'/case_'+str(i+1).zfill(3)
    for y_scale in ('linear', 'log'):
        path_y = path_case+'/'+y_scale
        if not os.path.exists(path_y):
            os.makedirs(path_y)
        for j, name in enumerate(inputs.network.names):
            x_ = x[i][-1]
            y_ = output_ops(y[i], j)
            y_pred_ = output_ops(y_pred[i], j)
            testing.plot_test(
                path_y,
                x_,
                y_,
                y_pred_,
                ts_label[name],
                fig_name=name,
                labels=[r'$t$ [s]', r'$Y$'],
                scales=['log', y_scale]
            )

if FLAGS['one_pass']:
    utils.print_submain("Testing 'one-pass' inference")
    out = jl.Parallel(n_jobs=-1)(
        jl.delayed(single_scenario)(path_, i) for i in tqdm(range(n))
    )

## Speedup
Setting up

In [ ]:
path_ = path + '/speedup/'
if not os.path.exists(path_):
    os.makedirs(path_)

t_end = 1.e-7
num_ics = [int(10**i) for i in range(5)]
labels = ['DeepONet', 'Numerical Solver']

In [ ]:
if FLAGS['speedup']:
    utils.print_submain("Plotting execution time")

    # ODE solver
    dataGen.ODE_sol.t_end = t_end
    test_data = dataGen.testing_data(
        inputs.data.file['test'],
        exec_time=True,
        header=inputs.data.header
    )
    exec_time = []
    for scenario in test_data:
        exec_time.append(scenario[-1])
    ode_time_ref = np.mean(np.array(exec_time))

    # DeepONet & comparison
    exec_time = []
    for n in tqdm(num_ics):
        x_ = [i[np.random.choice(i.shape[0], n, replace=True)] for i in x[-1]]
        start = time.time()
        y_pred = model.net.predict(x_)
#         y_pred = model.net.call(x_)
        end = time.time()
        nn_time = end - start
        ode_time = ode_time_ref * n
        exec_time.append(np.array([nn_time, ode_time]))
    exec_time = np.vstack(exec_time)
    speedup = exec_time[:,1] / exec_time[:,0]

    # Plot style
    style = dict(
        marker='s',
        fillstyle='full',
        markersize=5,
        ls='--',
        lw=1.5
    )

    # Plot execution times
    fig = plt.figure()
    plt.xlabel('\# ODEs solved')
    plt.ylabel('Time [s]')
    plt.loglog(num_ics, exec_time, label=labels,**style)
    plt.legend(fontsize=16)
    plt.show()
    fig.savefig(path_ + 'times.pdf')
    plt.close()

    # Plot speedup
    fig = plt.figure()
    plt.xlabel('\# ODEs solved')
    plt.ylabel('Speedup')
    plt.semilogx(num_ics, speedup, **style)
    fig.savefig(path_ + 'speedup.pdf')
    plt.show()
    plt.close()

## Recursive inference
Setting up

In [ ]:
path_ = path + '/recursiveInference/'
if not os.path.exists(path_):
    os.makedirs(path_)

ts_label = {}
for i, name in enumerate(inputs.network.names):
    d = 1 if i == 0 else inputs.general.dim['out'][i]
    ts_label[name] = [r'$O$'] \
        + [r'$O_2^{({%s},{%s})}$' % (i+1,j+1) for j in range(d)]

Recursive inference solver

In [ ]:
def recursive_inference(
    ic,
    dt0=1.e-7,
    t_end=1.e-3,
    dt_str=1.4,
    dt_max=1.e-4
):
    # Initialize variables
    T = np.array([[0.]])
    Y = tf.concat(
        [1. - np.sum(ic[0:1,2:], keepdims=True), ic[0:1,2:]], axis=1
    )
    # Initialize loop
    t = T
    dt = dt0
    while t <= t_end:
        # Update inputs
        ic_ = np.expand_dims(tf.concat([ic[0,:2], Y[-1,1:]], axis=0), axis=0)
        t_ = np.array([[dt]])
        # Predictions
        y = output_ops(model.net([ic_,t_]), timescale=3)
        # Update time step
        t = t + dt
        dt = min(dt*dt_str, dt_max)
        # Collect data
        T = np.vstack((T, t))
        Y = np.vstack((Y, y))
    return T, Y

Recursive inference plotter

In [ ]:
def plot_recursive_inference(
    path,
    x,
    y,
    x_pred,
    y_pred,
    var_names=None,
    fig_name=None,
    labels=[r'$t$ [s]', r'$Y$'],
    scales=['log', 'linear'],
    xlim=None,
    ylim=None
):
    fig = plt.figure()

    # Set up figure
    x_label, y_label = labels
    x_scale, y_scale = scales

    # X axis
    if x_label is not None:
        plt.xlabel(x_label)
    if x_scale is not None:
        plt.xscale(x_scale)
    if xlim is not None:
        plt.xlim(xlim)
    else:
        plt.xlim([min(x_pred), max(x_pred)])

    # Y axis
    if y_label is not None:
        plt.ylabel(y_label)
    if y_scale == 'log':
        plt.yscale(y_scale)
        if ylim is None:
            plt.ylim([np.amin(y)*5.e-1, np.amax(y)*5.e+0])
    else:
        delta = np.amax(y)*0.1
        if ylim is None:
            plt.ylim([np.amin(y)-delta, np.amax(y)+delta])
    if ylim is not None:
        plt.ylim(ylim)

    if len(y.shape) < 2:
        y = np.expand_dims(y, 1)

    # Plot figure
    # Define styles
    true_style = dict(
        lw=1.
    )
    pred_style = dict(
        marker='x',
        fillstyle='none',
        markersize=5,
        ls='',
        lw=1.
    )

    # Define parameters
    colors = []
    n = y.shape[1]

    # Loop over features
    for d in range(n):

        # >> Plot true values
        plt.plot(
            x,
            y[:,d],
            **true_style
        )
        colors.append(plt.gca().lines[-1].get_color())

        # >> Plot predicted values
        plt.plot(
            x_pred,
            y_pred[:,d],
            **pred_style,
            label=var_names[d],
            c=colors[-1]
        )

    # Plot color legends
    if n > 12:
        fs = 'x-small'
    elif n > 6:
        fs = 'small'
    else:
        fs = 'medium'
    plt.legend(
        [plt.plot([], [], c=colors[i])[0] for i in range(len(var_names))],
        var_names,
        fontsize=fs,
        ncol=int(np.ceil(n/3))
    )

    # Save figure
    fig_path = path + '/' + fig_name + '.pdf'
    fig.savefig(fig_path)
    plt.close()

In [ ]:
if FLAGS['recursive']:
    utils.print_submain("Testing 'recursive' inference")
    for i in tqdm(range(n)):
        # Get solution
        t_ = x[i][1]
        y_ = output_ops(y[i], timescale=-1)
        t_pred_, y_pred_ = recursive_inference(x[i][0])
        # Plot solution
        path_case = path_+'/case_'+str(i+1).zfill(3)
        if not os.path.exists(path_case):
            os.makedirs(path_case)
        for y_scale in ('linear', 'log'):
            plot_recursive_inference(
                path_case,
                t_,
                y_,
                t_pred_,
                y_pred_,
                list(ts_label.values())[-1],
                fig_name=y_scale,
                scales=['log', y_scale],
                xlim=[1.e-8, 1.e-3]
            )

## Nonequilibrium distance
A linear form for the distribution function, which results in a thermalized local Boltzmann distribution within individual bins, has been employed.

This function is defined as follows
\begin{equation*}
    \mathcal{F}_i\left(\epsilon_s\right):\qquad\log\left(\frac{g_s}{n_s}\right)=\alpha_i+\beta_i\epsilon_s
    \quad\forall s\in\mathcal{I}_i
\end{equation*}
where
\begin{align}
    \alpha_i&=\log\left(\frac{Q_i}{n_i}\right)\label{eq:alpha}\\
    \beta_i&=\frac{1}{k_BT_i}\nonumber\\
    n_i&=Y_i\frac{\rho}{m}\nonumber
\end{align}

Setting up

In [ ]:
path_ = path + '/noneq/'
if not os.path.exists(path_):
    os.makedirs(path_)

# Timescale to grouping strategies
grouping_list = ['RVE1', 'RVE3', 'RVE9', 'RVE27']

# Nonequilibrium distance names
noneq_dist_names = [
    [r'$\epsilon_{%s}$' % (j+1) for j in range(d)] for d in [1, 3, 9]
]

Initialize nonequilibrium controller

In [ ]:
from src_post.noneq_controller import NoneqController

In [ ]:
neq_control = NoneqController(
    grouping_list=grouping_list,
    ts_fast_conn=model.net.ts_fast_conn,
    species=inputs.general.species
)

### Nonequilibrium distance definition: OPTION 1
Consider the zero order truncation of the Maximum Entropy Principle reconstruction
\begin{align*}
    f_i=\alpha_i&=\log\left(\frac{Q_i}{n_i}\right)\\
    &=\log\left(\frac{Q_i}{Y_i}\frac{m}{\rho}\right)
\end{align*}

Nonequilibrium distance calculated as norm $\left(p=1\right)$ of the differential offset
* TS$^{(3)}$-to-TS$^{(4)}$
\begin{equation*}
    \epsilon_i^{(3-4)} = \sum_{j,\mathcal{I}_j^{(4)}\in\mathcal{I}_i^{(3)}}
    \left\lVert f_i^{(3)}-f_j^{(4)}\right\rVert_p
    \quad\forall\mathcal{I}_i^{(3)}\in\text{RVE}_9
\end{equation*}
* TS$^{(2)}$-to-TS$^{(4)}$
\begin{equation*}
    \epsilon_i^{(2-4)} = \sum_{j,\mathcal{I}_j^{(4)}\in\mathcal{I}_i^{(2)}}
    \left\lVert f_i^{(2)}-f_j^{(4)}\right\rVert_p
    \quad\forall\mathcal{I}_i^{(2)}\in\text{RVE}_3
\end{equation*}
* TS$^{(1)}$-to-TS$^{(4)}$
\begin{equation*}
    \epsilon_1^{(1-4)} = \sum_{j,\mathcal{I}_j^{(4)}\in\mathcal{I}_1^{(1)}}
    \left\lVert f_1^{(1)}-f_j^{(4)}\right\rVert_p
    \quad\mathcal{I}_1^{(1)}\in\text{RVE}_1
\end{equation*}

### Nonequilibrium distance definition: OPTION 2
Consider the first order truncation of the Maximum Entropy Principle reconstruction
\begin{align*}
    f_i(\epsilon_s)=\alpha_i+\beta_i\epsilon_s&=\log\left(\frac{Q_i}{n_i}\right)+\frac{\epsilon_s}{k_BT_i}\\
    &=\log\left(\frac{Q_i}{Y_i}\frac{m}{\rho}\right)+\frac{\epsilon_s}{k_BT_i}
\end{align*}

Nonequilibrium distance calculated as norm $\left(p=1\right)$ of the differential offset
* TS$^{(3)}$-to-TS$^{(4)}$
\begin{equation*}
    \epsilon_i^{(3-4)} = \sum_{j,\mathcal{I}_j^{(4)}\in\mathcal{I}_i^{(3)}}
    \left\lVert f_i(\epsilon_s)^{(3)}-f_j(\epsilon_s)^{(4)}\right\rVert_p
    \quad\wedge s\in\mathcal{I}_j^{(4)}
    \qquad\forall\mathcal{I}_i^{(3)}\in\text{RVE}_9
\end{equation*}
* TS$^{(2)}$-to-TS$^{(4)}$
\begin{equation*}
    \epsilon_i^{(2-4)} = \sum_{j,\mathcal{I}_j^{(4)}\in\mathcal{I}_i^{(2)}}
    \left\lVert f_i(\epsilon_s)^{(2)}-f_j(\epsilon_s)^{(4)}\right\rVert_p
    \quad\wedge s\in\mathcal{I}_j^{(4)}
    \qquad\forall\mathcal{I}_i^{(2)}\in\text{RVE}_3
\end{equation*}
* TS$^{(1)}$-to-TS$^{(4)}$
\begin{equation*}
    \epsilon_1^{(1-4)} = \sum_{j,\mathcal{I}_j^{(4)}\in\mathcal{I}_1^{(1)}}
    \left\lVert f_1(\epsilon_s)^{(1)}-f_j(\epsilon_s)^{(4)}\right\rVert_p
    \quad\wedge s\in\mathcal{I}_j^{(4)}
    \qquad\mathcal{I}_1^{(1)}\in\text{RVE}_1
\end{equation*}

Nonequilibrium distance plotter

In [ ]:
def plot_noneq_dist(
    x,
    y,
    x_pred=None,
    y_pred=None,
    var_names=None,
    fig_name=None,
    labels=[r'$t$ [s]', r'$\epsilon$'],
    scales=['log', 'linear'],
    title=None,
    xlim=None,
    ylim=None,
    save=True,
    path=None,
    fig_ax=None
):
    if fig_ax is None:
        fig, ax = plt.subplots()
    else:
        fig, ax = fig_ax

    # Set up figure -----------------------------------------------------------
    if title is not None:
        ax.set_title(title)
    x_label, y_label = labels
    x_scale, y_scale = scales
    # X axis
    if x_label is not None:
        ax.set_xlabel(x_label)
    if x_scale is not None:
        ax.set_xscale(x_scale)
    if xlim is not None:
        ax.set_xlim(xlim)
    elif x_scale != 'log':
        ax.set_xlim([min(x), max(x)])
    # Y axis
    if y_label is not None:
        ax.set_ylabel(y_label)
    if y_scale == 'log':
        ax.set_yscale(y_scale)
        if ylim is None:
            ax.set_ylim([np.amin(y)*5.e-1, np.amax(y)*5.e+0])
    else:
        delta = np.amax(y)*0.1
        if ylim is None:
            ax.set_ylim([np.amin(y)-delta, np.amax(y)+delta])
    if ylim is not None:
        ax.set_ylim(ylim)
    # Check 2nd dim: looping over columns
    if len(y.shape) < 2:
        y = np.expand_dims(y, 1)

    # Plot figure -------------------------------------------------------------
    # Define styles
    true_style = dict(
        lw=1.
    )
    pred_style = dict(
        marker='x',
        fillstyle='none',
        markersize=5,
        ls='',
        lw=1.
    )
    # Loop over columns
    n = y.shape[1]
    for d in range(n):
        # >> True values
        ax.plot(
            x,
            y[:,d],
            c=colors[d],
            **true_style
        )
        # >> Predicted values
        if y_pred is not None:
            ax.plot(
                x_pred,
                y_pred[:,d],
                label=var_names[d],
                c=colors[d],
                **pred_style,
            )
    # Add legend
    if var_names is not None:
        if n > 12:
            fs = 'x-small'
        elif n > 6:
            fs = 'small'
        else:
            fs = 'medium'
        ax.legend(
            [plt.plot([], [], c=colors[i])[0] for i in range(len(var_names))],
            var_names,
            fontsize=fs,
            ncol=int(np.ceil(n/3)),
            loc='upper right'
        )

    # Save figure -------------------------------------------------------------
    if save:
        # Save figure
        fig_path = path + '/' + fig_name + '.pdf'
        fig.savefig(fig_path)
        
    if fig_ax is not None:
        return fig, ax
            

## Coarse-grained levels distribution
Setting up

In [ ]:
path_ = path + '/levelsDist/'
if not os.path.exists(path_):
    os.makedirs(path_)

# Get reactive mixture thermodynamics
ts_mixtures = [
    mixMdl.multiComponent.MultiComponent(spThs) \
        for spThs in neq_control.species_thermos
]

# Time instants to evaluate levels distribution
time_shots = [10.**(-i) for i in range(12,1,-2)]

Compute the levels distribution dynamics for each timescale

In [ ]:
def get_ts_lev_dist(data):
    # Extract data
    (ic, t), Y = data

    # Get temperature and density from initial condition
    T = np.take(ic, [0], axis=1)
    rho = np.take(ic, [1], axis=1)

    # Extract O and O2 mass fraction from Timescale 1
    Y_O = np.take(Y[0], [1], axis=1)
    Y[0] = np.take(Y[0], [0], axis=1)

    # Loop over the various timescale mixtures
    ts_lev_dist = []
    mix = dict(O=Y_O)
    for i, ts_mix in enumerate(ts_mixtures):
        mix['O2'] = Y[i]
        # Update the mixture with the new mass fractions
        ts_mix.update(mix)
        ts_mix.n_(rho=rho)
        # Update the partition functions
        for spTh in ts_mix.spTh.values():
            spTh.intPF.update(T)
        # Get n, g, E levels for each bin
        ts_lev_dist.append(
            ts_mix.spTh['O2'].intPF.levels_distribution()
        )
    return t, ts_lev_dist

Coarse-grained levels distribution plotter

In [ ]:
def plot_levels_dist(
    lev_dist=None,
    var_names=grouping_list,
    time_instant=None,
    title=None,
    labels=[r'$\epsilon_i$ [eV]', r'$n_i\//\/g_i$ [m$^{-3}$]'],
    scales=['linear', 'log'],
    xlim=None,
    ylim=None,
    fig_ax=None,
    save=True,
    path=None
):
    if fig_ax is None:
        fig, ax = plt.subplots()
    else:
        fig, ax = fig_ax

    # Set up figure -----------------------------------------------------------
    if title is not None:
        ax.title(title)
    x_label, y_label = labels
    x_scale, y_scale = scales
    # X axis
    if x_label is not None:
        ax.xlabel(x_label)
    if x_scale is not None:
        ax.xscale(x_scale)
    if xlim is not None:
        ax.xlim(xlim)
    # Y axis
    if y_label is not None:
        ax.ylabel(y_label)
    if x_scale is not None:
        ax.yscale(y_scale)
    if ylim is not None:
        ax.ylim(ylim)

    # Plot figure -------------------------------------------------------------
    style = dict(
        lw=1,
        marker='o',
        fillstyle='full',
        markersize=0.3,
        linestyle='none'
    )
    # Loop over Timescales
    for i, (n, g, E) in enumerate(lev_dist):
        # Loop over bins
        for b in range(len(n)):
            n_b = tf.reshape(n[b][idx,:], (-1,1))
            ax.plot(E[b], n_b/g[b], c=colors[i], **style)
    # Time instant
    if time_instant is not None:
        ax.text(
            0.75, 0.92,
            r'$t$ = %.1e s' % time_instant,
            transform=ax.transAxes,
            fontsize=14
        )
    # Add legend
    if var_names is not None:
        ax.legend(
            [ax.plot([], [], c=colors[i])[0] for i in range(len(var_names))],
            var_names
        )

    # Save figure -------------------------------------------------------------
    if save:
        fig.savefig(path)
        
    if fig_ax is not None:
        return fig, ax


In [ ]:
# Plotting levels ---------------------------------------------------------
if FLAGS['levels']:
    utils.print_submain("Plotting levels distributions")

    for i, data in enumerate(tqdm(test_data)):
        # Create folder
        path_test = path_+'/test_'+str(i+1).zfill(3)
        if not os.path.exists(path_test):
            os.makedirs(path_test)
        # Get levels distribution
        data = copy.deepcopy(data)
        t, ts_lev_dist = get_ts_lev_dist(data)
        # Loop over time instants
        for j, t_shot in enumerate(time_shots):
            path_tshots = path_test + '/shot_' + str(j+1)
            if not os.path.exists(path_tshots):
                os.makedirs(path_tshots)
            idx = np.argmin(np.absolute(t - t_shot), axis=0)
            # Plot levels distribution
            plot_levels_dist(
                lev_dist=ts_lev_dist,
                time_instant=t[idx],
                path=path_tshots+'/TS.pdf'
            )
            for k in range(len(ts_lev_dist)):
                plot_levels_dist(
                    lev_dist=[ts_lev_dist[k]],
                    var_names=[grouping_list[k]],
                    time_instant=t[idx],
                    path=path_tshots+'/TS'+str(k+1)+'.pdf'
                )
                plot_levels(
                    path_tshots+'/TS'+str(k+1)+'.pdf',
                    [TS[k]],
                    inp.postprocessing.timescales[k],
                    title='Bins population at t = %.3e s' % t[idx]
                )
                if k != 0:
                    plot_levels(
                        path_tshots+'/TS'+str(k-1)+str(k)+'.pdf',
                        TS[k-1:k+1],
                        inp.postprocessing.timescales[k-1:k+1],
                        title='Bins population at t = %.3e s' % t[idx]
                    )
            
            

        # Get number densities
        T, rho = x[i][0][0][:2]
        t = x[i][1]
        y_pred_ = y_pred[i]
        for j, t_shot in enumerate(inp.postprocessing.time_shots):
            idx = np.argmin(np.absolute(t - t_shot), axis=0)

        for j, t_shot in enumerate(time_shots):
            idx = np.argmin(np.absolute(t - t_shot), axis=0)

                

#     mix = {}
#     for gr in inp.postprocessing.groupings:
#         species = inp.general.species 
#         species['O2']['rovib']['grouping'] = gr
#         spTh = {
#             name: specieMdl.specieThermos.SpecieThermos(
#                 name,
#                 info,
#                 thermo='CoupledEnergyModes',
#                 EOS='PerfectGas',
#             ) for name, info in species.items()
#         }
#             mix[gr] = mixMdl.multiComponent.MultiComponent(spTh)

#         for i in tqdm(range(len(x))):
#             path_test = path_+'/test_'+str(i+1)
#             if not os.path.exists(path_test):
#                 os.makedirs(path_test)

                


#                 path_tshots = path_test + '/shot_' + str(j+1)
#                 if not os.path.exists(path_tshots):
#                     os.makedirs(path_tshots)

#                 # Plot number densities
#                 plot_levels(
#                     path_tshots+'/TS012.pdf',
#                     TS,
#                     inp.postprocessing.timescales,
#                     title='t = %.2e s' % t[idx]
#                 )

#                 for k in range(len(TS)):
#                     plot_levels(
#                         path_tshots+'/TS'+str(k)+'.pdf',
#                         [TS[k]],
#                         inp.postprocessing.timescales[k],
#                         title='Bins population at t = %.3e s' % t[idx]
#                     )
#                     if k != 0:
#                         plot_levels(
#                             path_tshots+'/TS'+str(k-1)+str(k)+'.pdf',
#                             TS[k-1:k+1],
#                             inp.postprocessing.timescales[k-1:k+1],
#                             title='Bins population at t = %.3e s' % t[idx]
#                         )

## Animations of the dynamics
Setting up

In [ ]:
frames = 100
path_ = path + '/animations/'
if not os.path.exists(path_):
    os.makedirs(path_)

Sample one test case

In [ ]:
k = np.random.randint(0, 1)
data = copy.deepcopy(test_data[k])

Levels dynamics

In [ ]:
t, ts_lev_dist = get_ts_lev_dist(data)

In [ ]:
# Set up figure parameters ----------------------------------------------------
mpl_animation = {
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 16,
    'legend.fontsize': 12
}
# -----------------------------------------------------------------------------

def init_lev_dist_lines(
    ax,
    var_names=grouping_list,
    title=None
):
    # Set up axes
    ax.set_xlabel(r'$\epsilon_i$ [eV]')
    ax.set_ylabel(r'$n_i\//\/g_i$ [m$^{-3}$]')
    # Define plot style
    style = dict(
        lw=1,
        marker='o',
        fillstyle='full',
        markersize=0.3,
        linestyle=''
    )
    # Add legend
    ax.legend(
        [ax.plot([], [], c=colors[i])[0] for i in range(len(var_names))],
        var_names,
        loc='lower left'
    )
    # Add title
    if title is not None:
        ax.set_title(title)
    # Initialize lines.Line2D objects
    lines = [
        ax.semilogy([], [], c=colors[i], **style)[0] \
            for i in range(len(var_names))
    ]
    return ax, lines

with plt.rc_context(mpl_animation):
    # Initialize a figure in which the graphs will be plotted
    fig, ax = plt.subplots()

    # Initialize levels distribution lines objects
    ax, lines = init_lev_dist_lines(ax)

    # Initialize text in ax
    t_text = ax.text(0.75, 0.92, '', transform=ax.transAxes, fontsize=14)

    def animate(frame):
        idx = int(frame/frames*len(t))
        # Write time instant
        t_text.set_text(r'$t$ = %.1e s' % t[idx])
        # Loop over Timescales
        for i, (n, g, E) in enumerate(ts_lev_dist):
            # Loop over bins
            x, y = [], []
            for b in range(len(n)):
                n_b = tf.reshape(n[b][idx,:], (-1,1))
                x.append(E[b])
                y.append(n_b/g[b])
            X, Y = list(map(lambda z: tf.concat(z, 0).numpy().squeeze(), [x, y]))
            lines[i].set_data(X.squeeze(), Y.squeeze())
        # Rescale axis limits
        ax.relim()
        ax.autoscale_view(tight=False, scalex=False)
        return lines

# # Get animation
# anim = FuncAnimation(
#     fig,
#     animate,
#     frames=frames,
#     blit=True
# )

# # Save animation
# anim.save(path_+'lev_dist.mp4', fps=10, dpi=600)

# # Display animation
# HTML(anim.to_jshtml())

Levels dynamics and nonequilibrium distance

In [ ]:
# Inputs
mep_order = 0
norm_order = 2
mean = True

In [ ]:
# Evaluate noneq. distance
t, noneq_dist = neq_control.eval_df_norm(
    data=data,
    norm_order=norm_order,
    mep_order=mep_order,
    mean=mean
)

In [ ]:
print(noneq_dist[-2][-500:])

In [ ]:
print(bool1.shape)
print(bool2.T.shape)

In [ ]:
tf.nest.map_structure(lambda x: x.shape, noneq_dist)
s = tf.nest.map_structure(lambda x: x[1], noneq_dist)
masks = tf.nest.map_structure(lambda x: x > 0.5, s)
print(masks)
max_n_bins = 27

empty = np.array([None]*max_n_bins)
# Loop over timescales
for i, masks_i in enumerate(masks):
    # Loop over groups in each timescale
    for j, masks_ij in enumerate(masks_i):
        if not masks_ij:
            m1 = empty == None
            m2 = np.isin(
                np.arange(max_n_bins), model.net.ts_fast_conn[i][j]
            )
            idx = np.where(np.logical_and(m1, m2))
            empty[idx] = i
empty[np.where(empty == None)] = i+1

print(empty)

In [ ]:
# Set up figure parameters ----------------------------------------------------
mpl_animation = {
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'legend.fontsize': 8,
    'figure.titlesize': 16,
    'figure.figsize': np.array([6.4, 4.8])*1.5
}
# -----------------------------------------------------------------------------

with plt.rc_context(mpl_animation):
    # Initialize a figure with 4 subplots
    fig, axs = plt.subplots(2,2)
    axs = axs.reshape(-1)
    plt.tight_layout()

    # Plot nonequilibrium distance dynamics (3/4 subplots)
    neqd_vlines = [] # Vertical lines for time evolution
    for i, neqd in enumerate(noneq_dist):
        axs[i] = plot_noneq_dist(
            t,
            neqd,
            var_names=noneq_dist_names[i],
            labels=[r'$t$ [s]', r'$\epsilon^{({%s}-4)}$' % (i+1)],
            title=f'Nonequilibrium distance: TS{i+1}-TS4',
            save=False,
            fig_ax=(fig, axs[i])
        )[1]
        axs[i].ticklabel_format(axis='y', style="sci", scilimits=(0,0))
        # Initialize vertical lines
        neqd_vlines.append(
            axs[i].axvline(0., *axs[i].get_ylim(), color='k', lw=0.5)
        )

    # Initialize levels distribution dynamics lines objects (4/4 subplots)
    axs[-1], lev_lines = init_lev_dist_lines(
        axs[-1],
        title='States distribution dynamics'
    )

    def animate(frame):
        idx = int(frame/frames*len(t))
        # Write time instant
        fig.suptitle(r'$t$ = %.1e s' % t[idx], fontsize=mpl_animation['figure.titlesize'])
        # Set vertical lines data for time evolution
        for vl in neqd_vlines:
            vl.set_xdata(t[idx])
        # Loop over Timescales
        for i, (n, g, E) in enumerate(ts_lev_dist):
            # Loop over bins
            x, y = [], []
            for b in range(len(n)):
                n_b = tf.reshape(n[b][idx,:], (-1,1))
                x.append(E[b])
                y.append(n_b/g[b])
            X, Y = list(map(lambda z: tf.concat(z, 0).numpy().squeeze(), [x, y]))
            lev_lines[i].set_data(X.squeeze(), Y.squeeze())
        # Rescale axis limits
        axs[-1].relim()
        axs[-1].autoscale_view(tight=True, scalex=False)
        return lev_lines + neqd_vlines

# Get animation
anim = FuncAnimation(
    fig,
    animate,
    frames=frames,
    blit=True
)

# Save animation
anim.save(path_+'lev_neq_dist.mp4', fps=10, dpi=300)

# Display animation
HTML(anim.to_jshtml())

Grouped-reconstructed rate coefficient
\begin{align*}
    \frac{dn_P}{dt}&=
    -\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{PQ}n_Pn_\text{O}
    +\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{QP}n_Qn_\text{O}
    -K^D_{P}n_Pn_\text{O}
    +K^R_{P}n_\text{O}^3
    \qquad\forall\hspace{1mm}\mathcal{I}_P\subset\mathcal{I}\\
    %&&\text{Multiply by }\mathcal{F}_p\mathcal{F}_q:\\
    \mathcal{F}_p\mathcal{F}_q\frac{dn_P}{dt}&=
    \mathcal{F}_p\mathcal{F}_q\left(-\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{PQ}n_Pn_\text{O}
    +\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{QP}n_Qn_\text{O}
    -K^D_{P}n_Pn_\text{O}
    +K^R_{P}n_\text{O}^3\right)
    \qquad\forall\hspace{1mm}\mathcal{I}_P\subset\mathcal{I}\\
    %&&\text{Since }n_p&=n_P\mathcal{F}_p\text{ and }n_q=n_Q\mathcal{F}_q:\\
    \frac{dn_p}{dt}\mathcal{F}_q&=
    -\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{PQ}n_p\mathcal{F}_qn_\text{O}
    +\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{QP}\mathcal{F}_pn_qn_\text{O}
    -K^D_{P}\mathcal{F}_qn_pn_\text{O}
    +K^R_{P}\mathcal{F}_p\mathcal{F}_qn_\text{O}^3
    \qquad\forall\hspace{1mm}\mathcal{I}_p\subset\mathcal{I}\\
    %&&\text{Sum over }\mathcal{I}_q:\\
    \sum_{\mathcal{I}_q\subset\mathcal{I}_Q}\frac{dn_p}{dt}\mathcal{F}_q&=
    \sum_{\mathcal{I}_q\subset\mathcal{I}_Q}\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{PQ}n_p\mathcal{F}_qn_\text{O}
    +\sum_{\mathcal{I}_q\subset\mathcal{I}_Q}\sum_{\mathcal{I}_Q\subset\mathcal{I}}K_{QP}\mathcal{F}_pn_qn_\text{O}
    -\sum_{\mathcal{I}_q\subset\mathcal{I}_Q}K^D_{P}\mathcal{F}_qn_pn_\text{O}
    +\sum_{\mathcal{I}_q\subset\mathcal{I}_Q}K^R_{P}\mathcal{F}_p\mathcal{F}_qn_\text{O}^3
    \qquad\forall\hspace{1mm}\mathcal{I}_p\subset\mathcal{I}\\
    \frac{dn_p}{dt}&=
    -\sum_{\mathcal{I}_q\subset\mathcal{I}}K_{PQ}\mathcal{F}_qn_pn_\text{O}
    +\sum_{\mathcal{I}_q\subset\mathcal{I}}K_{QP}\mathcal{F}_pn_qn_\text{O}
    -K^D_{P}n_pn_\text{O}+K^R_{P}\mathcal{F}_pn_\text{O}^3
    \qquad\forall\hspace{1mm}\mathcal{I}_p\subset\mathcal{I}\\
    \frac{dn_p}{dt}&=
    -\sum_{\mathcal{I}_q\subset\mathcal{I}}\bar{K}_{pq}n_pn_\text{O}
    +\sum_{\mathcal{I}_q\subset\mathcal{I}}\bar{K}_{qp}n_qn_\text{O}
    -\bar{K}^D_{p}n_pn_\text{O}+\bar{K}^R_{p}n_\text{O}^3
    \qquad\forall\hspace{1mm}\mathcal{I}_p\subset\mathcal{I}\\
\end{align*}
So
\begin{align*}
    \bar{K}_{pq}&=K_{PQ}\mathcal{F}_q\\
    \bar{K}_{qp}&=K_{QP}\mathcal{F}_p\\
    \bar{K}^D_{p}&=K_P^D\\
    \bar{K}^R_{p}&=K_P^R\mathcal{F}_p
\end{align*}

In [ ]:
from src_post.chemistry_model import ChemistryModel

In [ ]:
ts_chem_model = []
for i, spThs in enumerate(neq_control.species_thermos):
    pf = copy.deepcopy(inputs.general.processFlags)
    if i == 0:
        pf['excit'] = 0
    ts_chem_model.append(
        ChemistryModel(
            specieThermos=spThs,
            processFlags=pf
        )
    )

In [ ]:
# T = data[0][0][0:1,0:1]
T = data[0][0][0,0]
T.shape

In [ ]:
T = utils.reshape_to_2DArray(T)

for cm in ts_chem_model:
    cm.update(T)
    K = cm.K_(T)
    K = tf.nest.map_structure(lambda x: utils.reshape_to_2DArray(x), K)
    print(tf.nest.map_structure(lambda x: x.shape if x is not None else None, K))

In [ ]:
bool1 = np.array([[False, False, False, False, True, True]])


In [ ]:
np.logical_or(bool1.T, bool1)

In [ ]:
k = np.random.rand(1,3)

k_ = tf.experimental.numpy.kron(k,tf.ones((1,9)))
print(k_)

In [ ]:
empty = np.array([None]*27)
idx = [1,2,3]

empty[idx] = 1

In [ ]:
empty